# PRÁCTICA 10: 3D Scatter Plot con Sprites de Pokémon
## Estudiante: Uriel Abdallah Medina Torres
## Grupo: B
## Fecha: 08/08/2026
## Visualización 3D de Estadísticas de Pokémon con Sprites
## Objetivo: Crear un gráfico de dispersión 3D interactivo que muestre la relación entre 
## generación, tipo y promedio de estadísticas de Pokémon, incorporando sprites y filtros interactivos

# =============================================================================
# 1. IMPORTAR LIBRERÍAS NECESARIAS
# =============================================================================

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 2. CARGAR EL DATASET
# =============================================================================

# Contenido: Estadísticas de 801 Pokémon desde la 1ra hasta la 7ma generación
# Incluye: ID, nombre, tipos, HP, ataque, defensa, ataque especial, 
#          defensa especial, velocidad, generación, legendario, etc.

# Cargamos el dataset desde un archivo CSV

In [3]:
df = pd.read_csv('../Practice-05/Pokemon.csv')  # Asegurarse de tener el archivo en el directorio

print("Dataset cargado exitosamente")
print(f"Dimensiones del dataset: {df.shape}")

Dataset cargado exitosamente
Dimensiones del dataset: (805, 13)


# =============================================================================
# 3. INSPECCIÓN INICIAL DEL DATASET
# =============================================================================

In [4]:
print("\n--- PRIMERAS FILAS DEL DATASET ---")
print(df.head())

print("\n--- INFORMACIÓN DEL DATASET ---")
print(df.info())

print("\n--- ESTADÍSTICAS DESCRIPTIVAS BÁSICAS ---")
print(df.describe())

print("\n--- NOMBRES DE COLUMNAS ---")
print(df.columns.tolist())


--- PRIMERAS FILAS DEL DATASET ---
   #                   Name Type 1  Type 2  Total  HP  Attack  Defense  \
0  1              Bulbasaur  Grass  Poison    318  45      49       49   
1  2                Ivysaur  Grass  Poison    405  60      62       63   
2  3               Venusaur  Grass  Poison    525  80      82       83   
3  3               Venusaur  Grass  Poison    525  80      82       83   
4  3  VenusaurMega Venusaur  Grass  Poison    625  80     100      123   

   Sp. Atk  Sp. Def  Speed  Generation  Legendary  
0       65       65     45           1      False  
1       80       80     60           1      False  
2      100      100     80           1      False  
3      100      100     80           1      False  
4      122      120     80           1      False  

--- INFORMACIÓN DEL DATASET ---
<class 'pandas.DataFrame'>
RangeIndex: 805 entries, 0 to 804
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0

# =============================================================================
# 4. LIMPIEZA Y NORMALIZACIÓN DE DATOS
# =============================================================================

# 4.1 Normalizar nombres de columnas

In [5]:
print("\n--- NOMBRES DE COLUMNAS ORIGINALES ---")
print(df.columns.tolist())


--- NOMBRES DE COLUMNAS ORIGINALES ---
['#', 'Name', 'Type 1', 'Type 2', 'Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary']


# 4.2 Renombrar columnas para estandarizar

In [6]:
df = df.rename(columns={
    '#': 'id',
    'Name': 'name',
    'Type 1': 'type1',
    'Type 2': 'type2',
    'Total': 'total',
    'HP': 'hp',
    'Attack': 'attack',
    'Defense': 'defense',
    'Sp. Atk': 'sp_attack',
    'Sp. Def': 'sp_defense',
    'Speed': 'speed',
    'Generation': 'generation',
    'Legendary': 'is_legendary'
})

print("\n--- COLUMNAS RENOMBRADAS ---")
print(df.columns.tolist())


--- COLUMNAS RENOMBRADAS ---
['id', 'name', 'type1', 'type2', 'total', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed', 'generation', 'is_legendary']


# 4.3 Normalizar valores categóricos (tipos)

In [7]:
df['type1'] = df['type1'].astype(str).str.capitalize()
df['type2'] = df['type2'].astype(str).str.capitalize().replace('Nan', 'None').fillna('None')

print("\n--- TIPOS DE POKÉMON NORMALIZADOS ---")
print("Tipo 1:", df['type1'].unique()[:10])
print("Tipo 2:", df['type2'].unique()[:10])


--- TIPOS DE POKÉMON NORMALIZADOS ---
Tipo 1: <StringArray>
[   'Grass',     'Fire',    'Water',      'Bug',   'Normal',   'Poison',
 'Electric',   'Ground',    'Fairy', 'Fighting']
Length: 10, dtype: str
Tipo 2: <StringArray>
[  'Poison',     'None',   'Flying',   'Dragon',   'Ground',    'Fairy',
    'Grass', 'Fighting',  'Psychic',    'Steel']
Length: 10, dtype: str


# 4.4 Verificar tipos de datos y convertir si es necesario
# Asegurar que las columnas numéricas son de tipo float/int

In [8]:
numeric_cols = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed', 'generation']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"Columna '{col}' convertida a numérica")
    else:
        print(f"ADVERTENCIA: Columna '{col}' no encontrada")

Columna 'hp' convertida a numérica
Columna 'attack' convertida a numérica
Columna 'defense' convertida a numérica
Columna 'sp_attack' convertida a numérica
Columna 'sp_defense' convertida a numérica
Columna 'speed' convertida a numérica
Columna 'generation' convertida a numérica


# =============================================================================
# 5. TRATAMIENTO DE VALORES NULOS Y DUPLICADOS
# =============================================================================

In [9]:
print("\n--- ANTES DE LA LIMPIEZA ---")
print(f"Valores nulos por columna:\n{df.isnull().sum()}")
print(f"Registros duplicados: {df.duplicated().sum()}")



--- ANTES DE LA LIMPIEZA ---
Valores nulos por columna:
id              0
name            0
type1           0
type2           0
total           0
hp              0
attack          0
defense         0
sp_attack       0
sp_defense      0
speed           0
generation      0
is_legendary    0
dtype: int64
Registros duplicados: 3


# 5.1 Verificar valores nulos en columnas importantes
# Para este dataset, solo puede haber nulos en type2 (que ya rellenamos)
# y en algunas estadísticas si el CSV tiene problemas

# 5.2 Eliminar registros con nulos en columnas críticas

In [10]:
critical_cols = ['name', 'type1', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']
df_clean = df.dropna(subset=critical_cols)

# 5.3 Eliminar duplicados basados en nombre y número

In [11]:
df_clean = df_clean.drop_duplicates(subset=['name', 'id'])

print("\n--- DESPUÉS DE LA LIMPIEZA ---")
print(f"Valores nulos por columna:\n{df_clean.isnull().sum()}")
print(f"Registros duplicados: {df_clean.duplicated().sum()}")
print(f"Dimensiones finales: {df_clean.shape}")


--- DESPUÉS DE LA LIMPIEZA ---
Valores nulos por columna:
id              0
name            0
type1           0
type2           0
total           0
hp              0
attack          0
defense         0
sp_attack       0
sp_defense      0
speed           0
generation      0
is_legendary    0
dtype: int64
Registros duplicados: 0
Dimensiones finales: (800, 13)


# =============================================================================
# 6. SELECCIÓN DE VARIABLES ESTADÍSTICAS
# =============================================================================
# Justificación de variables seleccionadas:
# - HP: Puntos de vida, fundamental para la supervivencia
# - Attack: Capacidad de ataque físico
# - Defense: Capacidad de defensa física
# - Sp_Attack: Capacidad de ataque especial
# - Sp_Defense: Capacidad de defensa especial
# - Speed: Velocidad, determina quién ataca primero
# Estas 6 estadísticas representan las habilidades fundamentales de los Pokémon

# =============================================================================
# 7. CREACIÓN DE NUEVA COLUMNA: PROMEDIO DE ESTADÍSTICAS
# =============================================================================

In [12]:
stats_cols = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']
df_clean['promedio_estadisticas'] = df_clean[stats_cols].mean(axis=1)

# Redondear a 2 decimales

In [13]:
df_clean['promedio_estadisticas'] = df_clean['promedio_estadisticas'].round(2)

print("\n--- ESTADÍSTICAS DE PROMEDIO ---")
print(df_clean[['name', 'promedio_estadisticas']].head())


--- ESTADÍSTICAS DE PROMEDIO ---
                    name  promedio_estadisticas
0              Bulbasaur                  53.00
1                Ivysaur                  67.50
2               Venusaur                  87.50
4  VenusaurMega Venusaur                 104.17
5             Charmander                  51.50


# =============================================================================
# 8. ANÁLISIS ESTADÍSTICO DESCRIPTIVO
# =============================================================================

In [14]:
print("\n--- ANÁLISIS ESTADÍSTICO COMPLETO ---")
stats_summary = df_clean[stats_cols + ['promedio_estadisticas']].agg(['mean', 'median', 'min', 'max', 'std'])
print(stats_summary)


--- ANÁLISIS ESTADÍSTICO COMPLETO ---
                hp      attack     defense   sp_attack  sp_defense  \
mean     69.258750   79.001250   73.842500   72.820000   71.902500   
median   65.000000   75.000000   70.000000   65.000000   70.000000   
min       1.000000    5.000000    5.000000   10.000000   20.000000   
max     255.000000  190.000000  230.000000  194.000000  230.000000   
std      25.534669   32.457366   31.183501   32.722294   27.828916   

             speed  promedio_estadisticas  
mean     68.277500              72.517038  
median   65.000000              75.000000  
min       5.000000              30.000000  
max     180.000000             130.000000  
std      29.060474              19.993931  


# Estadísticas por tipo principal

In [15]:
print("\n--- ESTADÍSTICAS POR TIPO PRINCIPAL ---")
type_stats = df_clean.groupby('type1')[stats_cols + ['promedio_estadisticas']].mean().round(2)
print(type_stats)


--- ESTADÍSTICAS POR TIPO PRINCIPAL ---
             hp  attack  defense  sp_attack  sp_defense   speed  \
type1                                                             
Bug       56.88   70.97    70.72      53.87       64.80   61.68   
Dark      66.81   88.39    70.23      74.65       69.52   76.16   
Dragon    83.31  112.12    86.38      96.84       88.84   83.03   
Electric  59.80   69.09    66.30      90.02       73.70   84.50   
Fairy     74.12   61.53    65.71      78.53       84.71   48.59   
Fighting  69.85   96.78    65.93      53.11       64.70   66.07   
Fire      69.90   84.77    67.77      88.98       72.21   74.44   
Flying    70.75   78.75    66.25      94.25       72.50  102.50   
Ghost     64.44   73.78    81.19      79.34       76.47   64.34   
Grass     67.27   73.21    70.80      77.50       70.43   61.93   
Ground    73.78   95.75    84.84      56.47       62.75   63.91   
Ice       72.00   72.75    71.42      77.54       76.29   63.46   
Normal    77.28   73.

# =============================================================================
# 9. PREPARACIÓN DE VARIABLES PARA VISUALIZACIÓN
# =============================================================================

# 9.1 Ordenar generaciones correctamente

In [16]:
df_clean['generation'] = df_clean['generation'].astype(int)
gen_order = sorted(df_clean['generation'].unique())

# 9.2 Mapear generaciones a nombres descriptivos

In [17]:
gen_mapping = {1: 'Gen 1 (Kanto)', 2: 'Gen 2 (Johto)', 3: 'Gen 3 (Hoenn)',
               4: 'Gen 4 (Sinnoh)', 5: 'Gen 5 (Unova)', 6: 'Gen 6 (Kalos)',
               7: 'Gen 7 (Alola)'}
df_clean['generation_label'] = df_clean['generation'].map(gen_mapping)

# 9.3 Preparar tipos para colores
# Obtener lista de tipos únicos y asignar colores

In [18]:
unique_types = sorted(df_clean['type1'].unique())
color_palette = px.colors.qualitative.Set3 + px.colors.qualitative.Pastel

# Crear diccionario de colores por tipo

In [19]:
type_colors = {t: color_palette[i % len(color_palette)] for i, t in enumerate(unique_types)}

print("\n--- VARIABLES PREPARADAS ---")
print(f"Generaciones: {gen_order}")
print(f"Tipos disponibles: {len(unique_types)}")


--- VARIABLES PREPARADAS ---
Generaciones: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Tipos disponibles: 18


# =============================================================================
# 10. OBTENCIÓN DE SPRITES (URLs)
# =============================================================================
# Fuente de sprites: https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/
# O usando una API alternativa

In [20]:
def get_sprite_url(pokemon_id):
    """Obtiene la URL del sprite de un Pokémon dado su ID"""
    # Usamos el ID como número entero para la URL
    return f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/{int(pokemon_id)}.png"

# Añadir columna de sprite URL

In [21]:
df_clean['sprite_url'] = df_clean['id'].apply(get_sprite_url)

# Verificar algunas URLs

In [22]:
print("\n--- EJEMPLOS DE SPRITES ---")
sample_pokemon = df_clean.head(3)[['name', 'id', 'sprite_url']]
for _, row in sample_pokemon.iterrows():
    print(f"{row['name']} (ID: {row['id']}): {row['sprite_url']}")


--- EJEMPLOS DE SPRITES ---
Bulbasaur (ID: 1): https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/1.png
Ivysaur (ID: 2): https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/2.png
Venusaur (ID: 3): https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/3.png


# =============================================================================
# 11. CONSTRUCCIÓN DEL SCATTER PLOT 3D
# =============================================================================

# 11.1 Crear figura 3D

In [23]:
fig = go.Figure()

# 11.2 Agregar trazas por tipo para diferenciar colores

In [24]:
for type_name in unique_types:
    subset = df_clean[df_clean['type1'] == type_name]
    
    if len(subset) == 0:
        continue
    
    # Crear datos de hover personalizados
    hover_text = []
    for _, row in subset.iterrows():
        hover_text.append(
            f"<b>{row['name']}</b><br>" +
            f"Tipo: {row['type1']} / {row['type2']}<br>" +
            f"Generación: {row['generation_label']}<br>" +
            f"HP: {row['hp']}<br>" +
            f"Ataque: {row['attack']}<br>" +
            f"Defensa: {row['defense']}<br>" +
            f"At. Especial: {row['sp_attack']}<br>" +
            f"Def. Especial: {row['sp_defense']}<br>" +
            f"Velocidad: {row['speed']}<br>" +
            f"<b>Promedio: {row['promedio_estadisticas']}</b><br>" +
            f"<img src='{row['sprite_url']}' style='width:80px;height:80px;'>"
        )
    
    # Agregar trace
    fig.add_trace(go.Scatter3d(
        x=subset['generation'],
        y=subset['type1'],
        z=subset['promedio_estadisticas'],
        mode='markers',
        name=type_name,
        marker=dict(
            size=10,
            color=type_colors[type_name],
            opacity=0.8,
            symbol='circle',
            line=dict(width=1, color='DarkSlateGrey')
        ),
        text=hover_text,
        hoverinfo='text',
        hovertemplate='%{text}<extra></extra>'
    ))


# 11.3 FILTROS INTERACTIVOS PARA GRÁFICA 3D

In [25]:
generation_buttons = [
    dict(
        method="update",
        label="Todas",
        args=[{"visible": [True] * len(unique_types)}]
    )
]

for gen in gen_order:
    gen_label = gen_mapping[gen]
    visible = [type_name in df_clean[df_clean['generation'] == gen]['type1'].unique() 
               for type_name in unique_types]
    generation_buttons.append(
        dict(
            method="update",
            label=gen_label,
            args=[{"visible": visible}]
        )
    )

menu = dict(
    x=0.1,
    y=0.9,
    buttons=generation_buttons,
    bgcolor='rgba(255,255,255,0.8)',
    bordercolor='black',
    font=dict(size=12)
)

# 11.4 PERSONALIZACIÓN DEL DISEÑO GRÁFICA 3D


In [26]:
fig.update_layout(
    title=dict(
        text="<b>Distribución de Pokémon por Generación, Tipo y Promedio de Estadísticas</b><br>" +
             "<sup>Visualización 3D Interactiva con Sprites</sup>",
        font=dict(size=20, color='#2E4053')
    ),
    scene=dict(
        xaxis=dict(
            title="<b>Generación</b>",
            tickvals=gen_order,
            ticktext=[gen_mapping[g] for g in gen_order],
            tickangle=45,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='rgba(240, 240, 240, 0.8)'
        ),
        yaxis=dict(
            title="<b>Tipo Principal</b>",
            tickfont=dict(size=10),
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='rgba(240, 240, 240, 0.8)'
        ),
        zaxis=dict(
            title="<b>Promedio de Estadísticas</b>",
            range=[0, df_clean['promedio_estadisticas'].max() * 1.1],
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='rgba(240, 240, 240, 0.8)'
        ),
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.5),
            up=dict(x=0, y=0, z=1),
            center=dict(x=0, y=0, z=0)
        )
    ),
    legend=dict(
        title="<b>Tipo Pokémon</b>",
        x=1.02,
        y=1,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=10),
        itemsizing='constant',
        tracegroupgap=0
    ),
    updatemenus=[menu],
    width=1200,
    height=800,
    margin=dict(l=50, r=200, t=100, b=50),
    paper_bgcolor='rgba(245, 245, 245, 0.95)',
    plot_bgcolor='rgba(245, 245, 245, 0.95)'
)

In [27]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# =============================================================================
# 1. CARGAR Y PREPARAR LOS DATOS
# =============================================================================

# Cargar el dataset desde URL
url = "https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv"
df = pd.read_csv(url)

# Renombrar columnas
df = df.rename(columns={
    '#': 'id',
    'Name': 'name',
    'Type 1': 'type1',
    'Type 2': 'type2',
    'Total': 'total',
    'HP': 'hp',
    'Attack': 'attack',
    'Defense': 'defense',
    'Sp. Atk': 'sp_attack',
    'Sp. Def': 'sp_defense',
    'Speed': 'speed',
    'Generation': 'generation',
    'Legendary': 'is_legendary'
})

# Limpiar datos
df['type1'] = df['type1'].str.capitalize()
df['type2'] = df['type2'].str.capitalize().fillna('None')

# Eliminar duplicados
df = df.drop_duplicates(subset=['name', 'id'], keep='first')

# Calcular promedio de estadísticas
stats_cols = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']
df['promedio_estadisticas'] = df[stats_cols].mean(axis=1).round(2)

# Mapear generaciones
gen_mapping = {1: 'Gen 1', 2: 'Gen 2', 3: 'Gen 3',
               4: 'Gen 4', 5: 'Gen 5', 6: 'Gen 6'}
df['generation_label'] = df['generation'].map(gen_mapping)

# Crear URL de sprites
def get_sprite_url(pokemon_id):
    return f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/{int(pokemon_id)}.png"

df['sprite_url'] = df['id'].apply(get_sprite_url)

# =============================================================================
# 2. CREAR GRÁFICO 3D CON IMÁGENES EN HOVER
# =============================================================================

# Obtener tipos únicos
unique_types = sorted(df['type1'].unique())
color_palette = px.colors.qualitative.Set3 + px.colors.qualitative.Pastel
type_colors = {t: color_palette[i % len(color_palette)] for i, t in enumerate(unique_types)}

# Crear figura
fig = go.Figure()

# Agregar trazas por tipo
for type_name in unique_types:
    subset = df[df['type1'] == type_name]
    
    if len(subset) == 0:
        continue
    
    # Preparar datos para hover con imágenes
    hover_texts = []
    for _, row in subset.iterrows():
        hover_text = (
            f"<b>{row['name']}</b><br>"
            f"<img src='{row['sprite_url']}' style='width:150px;height:150px;'><br>"
            f"Tipo: {row['type1']} / {row['type2']}<br>"
            f"Generación: {row['generation_label']}<br>"
            f"HP: {row['hp']}<br>"
            f"Ataque: {row['attack']}<br>"
            f"Defensa: {row['defense']}<br>"
            f"At. Especial: {row['sp_attack']}<br>"
            f"Def. Especial: {row['sp_defense']}<br>"
            f"Velocidad: {row['speed']}<br>"
            f"<b>Promedio: {row['promedio_estadisticas']}</b>"
        )
        hover_texts.append(hover_text)
    
    fig.add_trace(go.Scatter3d(
        x=subset['generation'],
        y=subset['type1'],
        z=subset['promedio_estadisticas'],
        mode='markers',
        name=type_name,
        marker=dict(
            size=10,
            color=type_colors[type_name],
            opacity=0.8,
            symbol='circle',
            line=dict(width=1, color='black')
        ),
        text=hover_texts,
        hoverinfo='text',
        hovertemplate='%{text}<extra></extra>'
    ))

# =============================================================================
# 3. CONFIGURAR EL DISEÑO
# =============================================================================

fig.update_layout(
    title=dict(
        text="<b>Scatter 3D de Pokémon: Promedio de Estadísticas por Tipo y Generación</b><br>" +
             "<sup>Pasa el cursor sobre cualquier punto para ver la imagen del Pokémon</sup>",
        font=dict(size=20, color='#2E4053')
    ),
    scene=dict(
        xaxis=dict(
            title="<b>Generación</b>",
            tickvals=[1, 2, 3, 4, 5, 6],
            ticktext=['Gen 1', 'Gen 2', 'Gen 3', 'Gen 4', 'Gen 5', 'Gen 6'],
            tickangle=45,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='rgba(240, 240, 240, 0.9)'
        ),
        yaxis=dict(
            title="<b>Tipo</b>",
            tickfont=dict(size=11),
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='rgba(240, 240, 240, 0.9)'
        ),
        zaxis=dict(
            title="<b>Promedio de estadísticas</b>",
            range=[0, df['promedio_estadisticas'].max() * 1.1],
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='rgba(240, 240, 240, 0.9)'
        ),
        camera=dict(
            eye=dict(x=1.8, y=1.8, z=1.5),
            up=dict(x=0, y=0, z=1),
            center=dict(x=0, y=0, z=0)
        )
    ),
    legend=dict(
        title="<b>Tipo Pokémon</b>",
        x=1.02,
        y=1,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=10)
    ),
    width=1400,
    height=900,
    margin=dict(l=50, r=200, t=100, b=50),
    paper_bgcolor='rgba(245, 245, 245, 0.95)',
    plot_bgcolor='rgba(245, 245, 245, 0.95)',
    hovermode='closest'
)

# =============================================================================
# 4. MOSTRAR Y EXPORTAR
# =============================================================================

fig.show()
fig.write_html("scatter_3d_pokemon_con_imagenes.html")
print("✅ Gráfico exportado: scatter_3d_pokemon_con_imagenes.html")

✅ Gráfico exportado: scatter_3d_pokemon_con_imagenes.html



# 11.5 Agregar anotaciones para Pokémon destacados

In [28]:
highlighted_pokemon = df_clean.nlargest(10, 'promedio_estadisticas')
annotations = []
for _, row in highlighted_pokemon.iterrows():
    annotations.append(
        dict(
            x=row['generation'],
            y=row['type1'],
            z=row['promedio_estadisticas'],
            text=f"<b>{row['name']}</b>",
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor='red',
            font=dict(size=12, color='red'),
            xanchor='center',
            yanchor='bottom'
        )
    )

fig.update_layout(scene=dict(annotations=annotations))


# 11.6 Ajustar opacidad y tamaño de marcadores


In [29]:
for trace in fig.data:
    trace.marker.size = 12
    trace.marker.opacity = 0.7
    trace.marker.line = dict(width=1, color='DarkSlateGrey')


# 11.7 Agregar línea de tendencia (media general)


In [30]:
mean_value = df_clean['promedio_estadisticas'].mean()
fig.add_trace(go.Scatter3d(
    x=[min(gen_order), max(gen_order)],
    y=[df_clean['type1'].iloc[0], df_clean['type1'].iloc[0]],
    z=[mean_value, mean_value],
    mode='lines',
    name=f'Media Global ({mean_value:.2f})',
    line=dict(color='red', width=4, dash='dash'),
    opacity=0.5,
    showlegend=True
))

# =============================================================================
# 12. CONFIGURACIÓN DE SPRITES (Elemento interactivo)
# =============================================================================

# Agregar sprites como imágenes en el gráfico usando anotaciones
# Nota: Esto se logra mejor con un callback en Dash, pero para Plotly puro
# usaremos una aproximación con imágenes en el hover o con un segundo gráfico

# Para esta demostración, mostraremos los sprites en un gráfico separado
# o usando customdata para mostrar en el hover

# Crear columna con HTML para mostrar sprite en hover

In [31]:
def create_sprite_html(row):
    return f"<img src='{row['sprite_url']}' style='width:96px;height:96px;'>"

# Agregar columna para hover con sprite

In [32]:
df_clean['sprite_html'] = df_clean.apply(create_sprite_html, axis=1)

# =============================================================================
# 13. FILTROS INTERACTIVOS (Implementados con updatemenus)
# =============================================================================

# 13.1 Crear botones para filtrar por generación

In [33]:
generation_buttons = [
    dict(
        method="update",
        label="Todas",
        args=[{"visible": [True] * len(unique_types)}]
    )
]

for gen in gen_order:
    gen_label = gen_mapping[gen]
    visible = [type_name in df_clean[df_clean['generation'] == gen]['type1'].unique() 
               for type_name in unique_types]
    generation_buttons.append(
        dict(
            method="update",
            label=gen_label,
            args=[{"visible": visible}]
        )
    )

# 13.2 Crear menú desplegable

In [34]:
menu = dict(
    x=0.1,
    y=0.9,
    buttons=generation_buttons,
    bgcolor='rgba(255,255,255,0.8)',
    bordercolor='black',
    font=dict(size=12)
)

# =============================================================================
# 14. PERSONALIZACIÓN DEL DISEÑO
# =============================================================================

# 14.1 Crear figura para mostrar Pokémon con sus sprites

In [35]:
fig_sprites = go.Figure()

# 14.2 Seleccionar los Pokémon más representativos por tipo
# Tomamos los 3 mejores y 3 peores por tipo

In [36]:
pokemon_seleccionados = []

for type_name in unique_types:
    subset = df_clean[df_clean['type1'] == type_name]
    if len(subset) > 0:
        # Top 3 mejores por tipo
        top3 = subset.nlargest(3, 'promedio_estadisticas')
        # Bottom 3 por tipo
        bottom3 = subset.nsmallest(3, 'promedio_estadisticas')
        pokemon_seleccionados.extend(top3.to_dict('records'))
        pokemon_seleccionados.extend(bottom3.to_dict('records'))

 14.3 Crear DataFrame con los Pokémon seleccionados

In [37]:
df_sprites = pd.DataFrame(pokemon_seleccionados)
df_sprites = df_sprites.drop_duplicates(subset=['name'])

# 14.4 Crear el gráfico de sprites

In [38]:
type_order = sorted(unique_types)
type_coords = {t: i for i, t in enumerate(type_order)}
df_sprites['type_coord'] = df_sprites['type1'].map(type_coords)

hover_texts = []
for _, row in df_sprites.iterrows():
    hover_text = (
        f"<b>{row['name']}</b><br>"
        f"Tipo: {row['type1']} / {row['type2']}<br>"
        f"Generación: {row['generation_label']}<br>"
        f"HP: {row['hp']}<br>"
        f"Ataque: {row['attack']}<br>"
        f"Defensa: {row['defense']}<br>"
        f"Sp. Atk: {row['sp_attack']}<br>"
        f"Sp. Def: {row['sp_defense']}<br>"
        f"Velocidad: {row['speed']}<br>"
        f"<b>Promedio: {row['promedio_estadisticas']}</b><extra></extra>"
    )
    hover_texts.append(hover_text)

fig_sprites = go.Figure()
fig_sprites.add_trace(go.Scatter(
    x=df_sprites['generation'],
    y=df_sprites['type_coord'],
    mode='markers',
    marker=dict(size=1, opacity=0),
    text=hover_texts,
    hoverinfo='text',
    hovertemplate='%{text}',
    showlegend=False
))

sprite_size = 0.6
for _, row in df_sprites.iterrows():
    fig_sprites.add_layout_image(dict(
        source=row['sprite_url'],
        xref='x',
        yref='y',
        x=row['generation'],
        y=row['type_coord'],
        xanchor='center',
        yanchor='middle',
        sizex=sprite_size,
        sizey=sprite_size,
        sizing='contain',
        opacity=1,
        layer='above',
    ))

fig_sprites.update_layout(
    title=dict(
        text="<b>Pokémon Destacados por Tipo y su Promedio de Estadísticas</b><br>"
             "<sup>Visualización con Sprites de Pokémon</sup>",
        font=dict(size=20, color='#2E4053')
    ),
    xaxis=dict(
        title="<b>Generación</b>",
        tickvals=[1, 2, 3, 4, 5, 6],
        ticktext=['Gen 1', 'Gen 2', 'Gen 3', 'Gen 4', 'Gen 5', 'Gen 6'],
        tickangle=45,
        gridcolor='lightgray',
        showgrid=True
    ),
    yaxis=dict(
        title="<b>Tipo Principal</b>",
        tickvals=list(type_coords.values()),
        ticktext=list(type_order),
        gridcolor='lightgray',
        showgrid=True,
        range=[-1, len(type_order)],
    ),
    width=1400,
    height=900,
    margin=dict(l=50, r=50, t=100, b=150),
    paper_bgcolor='rgba(245, 245, 245, 0.95)',
    plot_bgcolor='rgba(245, 245, 245, 0.95)',
    showlegend=False,
    hovermode='closest'
)

fig_sprites.show()


In [39]:
# =============================================================================
# VISUALIZACIÓN 3D INTERACTIVA Y EXPORTACIÓN PYDECK CON SPRITES
# =============================================================================
import pydeck as pdk

# 1. Generar la gráfica 3D en Pydeck con sprites livianos
def get_fast_sprite_url(pokemon_id):
    return f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{int(pokemon_id)}.png"

df_clean['sprite_fast_url'] = df_clean['id'].apply(get_fast_sprite_url)

df_clean['icon_data'] = df_clean['sprite_fast_url'].apply(lambda url: {
    "url": url,
    "width": 96,
    "height": 96,
    "anchorY": 48,
    "anchorX": 48
})

type_coords_pydeck = {t: i * 40 for i, t in enumerate(unique_types)}
df_clean['x_3d'] = df_clean['generation'] * 100
df_clean['y_3d'] = df_clean['type1'].map(type_coords_pydeck)
df_clean['z_3d'] = df_clean['promedio_estadisticas'] * 2.5

icon_layer = pdk.Layer(
    "IconLayer",
    data=df_clean,
    get_icon="icon_data",
    get_position=["x_3d", "y_3d", "z_3d"],
    get_size=35,
    size_scale=1,
    pickable=True,
    billboard=True
)

view_state = pdk.ViewState(
    target=[df_clean['x_3d'].mean(), df_clean['y_3d'].mean(), df_clean['z_3d'].mean()],
    zoom=0.2,
    rotation_orbit=45,
    rotation_x=30
)

view = pdk.View(type="OrbitView", controller=True)

tooltip = {
    "html": "<b>{name}</b> (ID: {id})<br>" \
            "Tipo: {type1} / {type2}<br>" \
            "Generación: Gen {generation}<br>" \
            "<b>Promedio Estadísticas: {promedio_estadisticas}</b><br><br>" \
            "<img src='{sprite_fast_url}' style='width:80px; height:80px;' />",
    "style": {"backgroundColor": "rgba(25, 25, 25, 0.95)", "color": "white", "padding": "10px", "borderRadius": "8px"}
}

deck_chart = pdk.Deck(layers=[icon_layer], initial_view_state=view_state, views=[view], tooltip=tooltip)

# Guardar el archivo HTML autónomo para abrir en el navegador
html_filename = "pokemon_3d_pydeck.html"
deck_chart.to_html(html_filename)
print(f"Éxito: Gráfica Pydeck 3D exportada a '{html_filename}'.")
print("Abra 'pokemon_3d_pydeck.html' en su navegador para explorar los 800 sprites en 3D libre.")

# Mostrar también la gráfica 3D Plotly nativa (100% compatible con VS Code Notebook)
fig.show()


Éxito: Gráfica Pydeck 3D exportada a 'pokemon_3d_pydeck.html'.
Abra 'pokemon_3d_pydeck.html' en su navegador para explorar los 800 sprites en 3D libre.


In [40]:
# =============================================================================
# VISUALIZACIÓN 3D INTERACTIVA DIRECTA EN EL NOTEBOOK CON PYDECK
# =============================================================================
import pydeck as pdk

# 1. Generar la gráfica 3D en Pydeck con sprites livianos
def get_fast_sprite_url(pokemon_id):
    return f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{int(pokemon_id)}.png"

df_clean['sprite_fast_url'] = df_clean['id'].apply(get_fast_sprite_url)

df_clean['icon_data'] = df_clean['sprite_fast_url'].apply(lambda url: {
    "url": url,
    "width": 96,
    "height": 96,
    "anchorY": 48,
    "anchorX": 48
})

type_coords_pydeck = {t: i * 40 for i, t in enumerate(unique_types)}
df_clean['x_3d'] = df_clean['generation'] * 100
df_clean['y_3d'] = df_clean['type1'].map(type_coords_pydeck)
df_clean['z_3d'] = df_clean['promedio_estadisticas'] * 2.5

# 2. Configurar la capa de iconos (Sprites de Pokémon)
icon_layer = pdk.Layer(
    "IconLayer",
    data=df_clean,
    get_icon="icon_data",
    get_position=["x_3d", "y_3d", "z_3d"],
    get_size=35,
    size_scale=1,
    pickable=True,
    billboard=True
)

# 3. Configurar la vista de la cámara 3D
view_state = pdk.ViewState(
    target=[df_clean['x_3d'].mean(), df_clean['y_3d'].mean(), df_clean['z_3d'].mean()],
    zoom=0.2,
    rotation_orbit=45,
    rotation_x=30
)

view = pdk.View(type="OrbitView", controller=True)

# 4. Configurar el Tooltip interactivo con la imagen al pasar el cursor
tooltip = {
    "html": "<b>{name}</b> (ID: {id})<br>" \
            "Tipo: {type1} / {type2}<br>" \
            "Generación: Gen {generation}<br>" \
            "<b>Promedio Estadísticas: {promedio_estadisticas}</b><br><br>" \
            "<img src='{sprite_fast_url}' style='width:80px; height:80px;' />",
    "style": {"backgroundColor": "rgba(25, 25, 25, 0.95)", "color": "white", "padding": "10px", "borderRadius": "8px"}
}

# 5. Ensamblar el Deck
deck_chart = pdk.Deck(
    layers=[icon_layer], 
    initial_view_state=view_state, 
    views=[view], 
    tooltip=tooltip
)

# 6. Renderizar de forma nativa e interactiva dentro de la celda del notebook
deck_chart.show()

In [41]:
fig_sprites.update_layout(
    title=dict(
        text="<b>Pokémon Destacados por Tipo y su Promedio de Estadísticas</b><br>" +
             "<sup>Visualización con Sprites de Pokémon</sup>",
        font=dict(size=20, color='#2E4053')
    ),
    xaxis=dict(
        title="<b>Generación</b>",
        tickvals=[1, 2, 3, 4, 5, 6],
        ticktext=['Gen 1', 'Gen 2', 'Gen 3', 'Gen 4', 'Gen 5', 'Gen 6'],
        tickangle=45,
        gridcolor='lightgray',
        showgrid=True
    ),
    yaxis=dict(
        title="<b>Tipo Principal</b>",
        tickvals=list(type_coords.values()),
        ticktext=list(type_order),
        gridcolor='lightgray',
        showgrid=True,
        range=[-1, len(type_order)],
    ),
    width=1400,
    height=900,
    margin=dict(l=50, r=50, t=100, b=150),
    paper_bgcolor='rgba(245, 245, 245, 0.95)',
    plot_bgcolor='rgba(245, 245, 245, 0.95)',
    showlegend=False,
    hovermode='closest'
)


# 14.6 Agregar filtros por generación para el gráfico de sprites

In [42]:
fig_sprites.update_layout(
    title=dict(
        text="<b>Pokémon Destacados por Tipo y su Promedio de Estadísticas</b><br>" +
             "<sup>Visualización con Sprites de Pokémon</sup>",
        font=dict(size=20, color='#2E4053')
    ),
    xaxis=dict(
        title="<b>Generación</b>",
        tickvals=[1, 2, 3, 4, 5, 6],
        ticktext=['Gen 1', 'Gen 2', 'Gen 3', 'Gen 4', 'Gen 5', 'Gen 6'],
        tickangle=45,
        gridcolor='lightgray',
        showgrid=True
    ),
    yaxis=dict(
        title="<b>Tipo Principal</b>",
        tickvals=list(type_coords.values()),
        ticktext=list(type_order),
        gridcolor='lightgray',
        showgrid=True,
        range=[-1, len(type_order)],
    ),
    width=1400,
    height=900,
    margin=dict(l=50, r=50, t=100, b=150),
    paper_bgcolor='rgba(245, 245, 245, 0.95)',
    plot_bgcolor='rgba(245, 245, 245, 0.95)',
    showlegend=False,
    hovermode='closest'
)


# 14.7 CREAR SUBPLOTS CON AMBAS GRÁFICAS

In [43]:
# Crear figura 3D
fig_3d = go.Figure()

for type_name in sorted(df['type1'].unique()):
    subset = df[df['type1'] == type_name]
    if len(subset) == 0:
        continue
    
    hover_texts = []
    for _, row in subset.iterrows():
        hover_text = (
            f"<b>{row['name']}</b><br>"
            f"<img src='{row['sprite_url']}' style='width:120px;height:120px;'><br>"
            f"Tipo: {row['type1']} / {row['type2']}<br>"
            f"Generación: {row['generation']}<br>"
            f"<b>Promedio: {row['promedio_estadisticas']}</b>"
        )
        hover_texts.append(hover_text)
    
    fig_3d.add_trace(
        go.Scatter3d(
            x=subset['generation'],
            y=subset['type1'],
            z=subset['promedio_estadisticas'],
            mode='markers',
            name=type_name,
            marker=dict(
                size=8,
                opacity=0.6,
                line=dict(width=1, color='black')
            ),
            text=hover_texts,
            hoverinfo='text',
            hovertemplate='%{text}<extra></extra>'
        )
    )

# Configurar layout 3D
fig_3d.update_layout(
    title=dict(
        text="<b>Gráfico 3D: Distribución por Generación, Tipo y Promedio</b><br>" +
             "<sup>Pasa el cursor sobre cualquier punto para ver la imagen del Pokémon</sup>",
        font=dict(size=20, color='#2E4053')
    ),
    scene=dict(
        xaxis=dict(
            title="<b>Generación</b>",
            tickvals=[1, 2, 3, 4, 5, 6],
            ticktext=['Gen 1', 'Gen 2', 'Gen 3', 'Gen 4', 'Gen 5', 'Gen 6'],
            tickangle=45,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='rgba(240, 240, 240, 0.9)'
        ),
        yaxis=dict(
            title="<b>Tipo</b>",
            tickfont=dict(size=10),
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='rgba(240, 240, 240, 0.9)'
        ),
        zaxis=dict(
            title="<b>Promedio de estadísticas</b>",
            range=[0, df['promedio_estadisticas'].max() * 1.1],
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='rgba(240, 240, 240, 0.9)'
        ),
        camera=dict(
            eye=dict(x=1.8, y=1.8, z=1.5),
            up=dict(x=0, y=0, z=1),
            center=dict(x=0, y=0, z=0)
        )
    ),
    legend=dict(
        title="<b>Tipo Pokémon</b>",
        x=1.02,
        y=1,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=10)
    ),
    width=1400,
    height=800,
    margin=dict(l=50, r=200, t=100, b=50),
    paper_bgcolor='rgba(245, 245, 245, 0.95)',
    plot_bgcolor='rgba(245, 245, 245, 0.95)',
    hovermode='closest'
)

# Mostrar gráfico 3D
fig_3d.show()
fig_3d.write_html("grafico_3d_pokemon.html")
print("✅ Gráfico 3D exportado: grafico_3d_pokemon.html")

# =============================================================================
# GRÁFICO 2D: SPRITES VISIBLES DIRECTAMENTE
# =============================================================================

# Seleccionar Pokémon para mostrar (mejor de cada tipo)
pokemon_seleccionados = []
for type_name in sorted(df['type1'].unique()):
    subset = df[df['type1'] == type_name]
    if len(subset) > 0:
        best = subset.nlargest(1, 'promedio_estadisticas')
        pokemon_seleccionados.extend(best.to_dict('records'))

# Agregar top 10 globales
top_global = df.nlargest(10, 'promedio_estadisticas')
pokemon_seleccionados.extend(top_global.to_dict('records'))

df_selected = pd.DataFrame(pokemon_seleccionados).drop_duplicates(subset=['name'])

# Mapear tipos a coordenadas
type_order = sorted(df['type1'].unique())
type_coords = {t: i for i, t in enumerate(type_order)}
df_selected['type_coord'] = df_selected['type1'].map(type_coords)

# Crear figura 2D
fig_2d = go.Figure()

# Puntos invisibles para hover
hover_texts = []
for _, row in df_selected.iterrows():
    hover_text = (
        f"<b>{row['name']}</b><br>"
        f"<img src='{row['sprite_url']}' style='width:120px;height:120px;'><br>"
        f"Tipo: {row['type1']} / {row['type2']}<br>"
        f"Generación: {row['generation']}<br>"
        f"HP: {row['hp']}<br>"
        f"Ataque: {row['attack']}<br>"
        f"Defensa: {row['defense']}<br>"
        f"At. Especial: {row['sp_attack']}<br>"
        f"Def. Especial: {row['sp_defense']}<br>"
        f"Velocidad: {row['speed']}<br>"
        f"<b>Promedio: {row['promedio_estadisticas']}</b>"
    )
    hover_texts.append(hover_text)

fig_2d.add_trace(
    go.Scatter(
        x=df_selected['generation'],
        y=df_selected['type_coord'],
        mode='markers',
        marker=dict(size=1, opacity=0),
        text=hover_texts,
        hoverinfo='text',
        hovertemplate='%{text}<extra></extra>',
        showlegend=False
    )
)

# Agregar imágenes de los Pokémon
sprite_size = 0.6
for _, row in df_selected.iterrows():
    fig_2d.add_layout_image(dict(
        source=row['sprite_url'],
        xref='x',
        yref='y',
        x=row['generation'],
        y=row['type_coord'],
        xanchor='center',
        yanchor='middle',
        sizex=sprite_size,
        sizey=sprite_size,
        sizing='contain',
        opacity=1,
        layer='above',
    ))

# Configurar layout 2D
fig_2d.update_layout(
    title=dict(
        text="<b>Gráfico 2D: Pokémon Destacados con Sprites</b><br>" +
             "<sup>Los sprites de los Pokémon más destacados son visibles directamente</sup>",
        font=dict(size=20, color='#2E4053')
    ),
    xaxis=dict(
        title="<b>Generación</b>",
        tickvals=[1, 2, 3, 4, 5, 6],
        ticktext=['Gen 1', 'Gen 2', 'Gen 3', 'Gen 4', 'Gen 5', 'Gen 6'],
        tickangle=45,
        gridcolor='lightgray',
        showgrid=True
    ),
    yaxis=dict(
        title="<b>Tipo Principal</b>",
        tickvals=list(type_coords.values()),
        ticktext=list(type_order),
        gridcolor='lightgray',
        showgrid=True,
        range=[-0.5, len(type_order) - 0.5],
    ),
    width=1400,
    height=700,
    margin=dict(l=80, r=50, t=100, b=150),
    paper_bgcolor='rgba(245, 245, 245, 0.95)',
    plot_bgcolor='rgba(245, 245, 245, 0.95)',
    showlegend=False,
    hovermode='closest'
)

# Mostrar gráfico 2D
fig_2d.show()
fig_2d.write_html("grafico_2d_sprites.html")
print("✅ Gráfico 2D exportado: grafico_2d_sprites.html")

✅ Gráfico 3D exportado: grafico_3d_pokemon.html


✅ Gráfico 2D exportado: grafico_2d_sprites.html


# =============================================================================
# 15. MEJORAS ADICIONALES
# =============================================================================

# 15.1 Agregar anotaciones con sprites (versión simplificada)
# Seleccionar algunos Pokémon destacados para mostrar sprites

In [44]:
highlighted_pokemon = df_clean.nlargest(10, 'promedio_estadisticas')
annotations = []
for _, row in highlighted_pokemon.iterrows():
    annotations.append(
        dict(
            x=row['generation'],
            y=row['type1'],
            z=row['promedio_estadisticas'],
            text=f"<b>{row['name']}</b>",
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor='red',
            font=dict(size=12, color='red'),
            xanchor='center',
            yanchor='bottom'
        )
    )

fig.update_layout(scene=dict(annotations=annotations))

# 15.2 Ajustar opacidad y tamaño de marcadores

In [45]:
for trace in fig.data:
    trace.marker.size = 12
    trace.marker.opacity = 0.7
    trace.marker.line = dict(width=1, color='DarkSlateGrey')

# 15.3 Agregar línea de tendencia (media general)

In [46]:
mean_value = df_clean['promedio_estadisticas'].mean()
fig.add_trace(go.Scatter3d(
    x=[min(gen_order), max(gen_order)],
    y=[df_clean['type1'].iloc[0], df_clean['type1'].iloc[0]],
    z=[mean_value, mean_value],
    mode='lines',
    name=f'Media Global ({mean_value:.2f})',
    line=dict(color='red', width=4, dash='dash'),
    opacity=0.5,
    showlegend=True
))

# =============================================================================
# 16. IDENTIFICACIÓN DE PATRONES Y HALLAZGOS
# =============================================================================

In [47]:
print("\n" + "="*60)
print("HALLAZGOS RELEVANTES DEL ANÁLISIS")
print("="*60)


HALLAZGOS RELEVANTES DEL ANÁLISIS


# Hallazgo 1: Pokémon legendarios tienden a tener promedios más altos

In [48]:
legendary_pokemon = df_clean[df_clean['is_legendary'] == True]['promedio_estadisticas'].mean()
non_legendary = df_clean[df_clean['is_legendary'] == False]['promedio_estadisticas'].mean()
print(f"\n1. Pokémon Legendarios vs No Legendarios:")
print(f"   - Promedio Legendarios: {legendary_pokemon:.2f}")
print(f"   - Promedio No Legendarios: {non_legendary:.2f}")
print(f"   - Diferencia: {legendary_pokemon - non_legendary:.2f} puntos")


1. Pokémon Legendarios vs No Legendarios:
   - Promedio Legendarios: 106.23
   - Promedio No Legendarios: 69.54
   - Diferencia: 36.70 puntos


# Hallazgo 2: Tipos con mejor promedio

In [49]:
best_types = df_clean.groupby('type1')['promedio_estadisticas'].mean().sort_values(ascending=False).head(5)
print(f"\n2. Top 5 Tipos con mejor promedio de estadísticas:")
for type_name, avg in best_types.items():
    print(f"   - {type_name}: {avg:.2f}")


2. Top 5 Tipos con mejor promedio de estadísticas:
   - Dragon: 91.75
   - Steel: 81.28
   - Flying: 80.84
   - Psychic: 79.32
   - Fire: 76.35


# Hallazgo 3: Evolución por generación

In [50]:
gen_stats = df_clean.groupby('generation')['promedio_estadisticas'].mean()
print(f"\n3. Evolución del promedio por generación:")
for gen, avg in gen_stats.items():
    print(f"   - {gen_mapping[gen]}: {avg:.2f}")
print(f"   - Tendencia: {'Aumento' if gen_stats.iloc[-1] > gen_stats.iloc[0] else 'Disminución'}")


3. Evolución del promedio por generación:
   - Gen 1: 71.14
   - Gen 2: 69.71
   - Gen 3: 72.70
   - Gen 4: 76.50
   - Gen 5: 72.50
   - Gen 6: 72.73
   - Tendencia: Aumento


# =============================================================================
# 17. EXPORTACIÓN A HTML
# =============================================================================

In [51]:
fig.write_html("pokemon_3d_scatter_plot.html")
print("\n" + "="*60)
print("GRÁFICO EXPORTADO EXITOSAMENTE")
print("Archivo: pokemon_3d_scatter_plot.html")
print("="*60)


GRÁFICO EXPORTADO EXITOSAMENTE
Archivo: pokemon_3d_scatter_plot.html


# =============================================================================
# 18. CONCLUSIONES FINALES
# =============================================================================

In [52]:
print("\n" + "="*60)
print("CONCLUSIONES FINALES")
print("="*60)

print("""
1. El gráfico 3D interactivo permite visualizar la relación entre generación, 
   tipo y promedio de estadísticas de Pokémon de manera intuitiva.

2. Se observan agrupaciones claras por tipo, donde tipos como Dragón y Psíquico
   tienden a tener promedios más altos, mientras que tipos como Bicho y Planta
   suelen tener promedios más bajos.

3. La integración de sprites y filtros interactivos mejora significativamente
   la experiencia de exploración de datos, permitiendo identificar patrones
   específicos por generación y tipo.

4. El análisis revela que los Pokémon legendarios y pseudo-legendarios (como
   Mewtwo, Rayquaza, etc.) sobresalen como outliers con promedios muy altos.

5. Se observa una tendencia general al aumento del promedio de estadísticas
   en generaciones más recientes, lo que podría indicar un power creep en el
   diseño de Pokémon.

6. La herramienta desarrollada es útil para investigadores, jugadores y
   diseñadores de juegos para entender la distribución y evolución de las
   estadísticas de Pokémon a lo largo de las generaciones.
""")

print("\n" + "="*60)
print("PRÁCTICA COMPLETADA EXITOSAMENTE")
print("="*60)


CONCLUSIONES FINALES

1. El gráfico 3D interactivo permite visualizar la relación entre generación, 
   tipo y promedio de estadísticas de Pokémon de manera intuitiva.

2. Se observan agrupaciones claras por tipo, donde tipos como Dragón y Psíquico
   tienden a tener promedios más altos, mientras que tipos como Bicho y Planta
   suelen tener promedios más bajos.

3. La integración de sprites y filtros interactivos mejora significativamente
   la experiencia de exploración de datos, permitiendo identificar patrones
   específicos por generación y tipo.

4. El análisis revela que los Pokémon legendarios y pseudo-legendarios (como
   Mewtwo, Rayquaza, etc.) sobresalen como outliers con promedios muy altos.

5. Se observa una tendencia general al aumento del promedio de estadísticas
   en generaciones más recientes, lo que podría indicar un power creep en el
   diseño de Pokémon.

6. La herramienta desarrollada es útil para investigadores, jugadores y
   diseñadores de juegos para entende

# Mostrar el gráfico en el notebook

In [53]:
fig.show()